# 09_02 Generalised Linear Models: Logistic, Poisson and Gamma Regression

**Learning objectives**
- State the three parts of a **GLM**: a distribution from the exponential family, a **linear predictor** $\eta=X\beta$ and a **link** $g(\mu)=\eta$.
- Fit logistic, Poisson and Gamma regressions with **iteratively reweighted least squares (IRLS)** written from scratch, and read the coefficients as **odds ratios** and **rate ratios**.
- Compute **standard errors** and **Wald intervals** from the Fisher information, use the **deviance**, handle **exposure** with an offset, and detect **overdispersion** (negative binomial fix).
- Choose a link by comparing models (logit, probit, complementary log-log) and see where the linear probability model fails.

**Prerequisites:** [09_01 Losses as likelihoods](09_01_losses_as_likelihoods.ipynb), [06_10 Wald, score and likelihood-ratio tests](../../module_06_inference/notebooks/06_10_wald_score_lrt.ipynb), [01_01 Bernoulli and Binomial](../../module_01_discrete/notebooks/01_01_bernoulli_binomial.ipynb), [01_04 Poisson](../../module_01_discrete/notebooks/01_04_poisson.ipynb).

## 1. Setup

All imports in one cell; the generator is seeded so every run gives the same numbers.

In [ ]:
import sys
sys.path.append("../..")  # repo root, so common.style can be imported
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
from scipy import stats
from common.style import apply_style, MAIN, SIM, HIGHLIGHT, NEUTRAL, COLORS

apply_style()
rng = np.random.default_rng(42)  # fixed seed so every run gives the same numbers

## 2. Prerequisite recap
Linear regression assumes $y\sim N(X\beta,\sigma^2)$. Maximum likelihood gives estimates with standard errors from the **Fisher information** $\mathcal I(\beta)$, and Wald, score and likelihood-ratio tests (06_10). A Bernoulli outcome has mean $p$ and variance $p(1-p)$; a Poisson count has mean and variance $\lambda$.

## 3. Intuition and story
Ordinary regression predicts a number that can be anything. But a conversion probability must stay in $[0,1]$, a number of claims cannot be negative, and the spread of a claim *amount* grows with its size. A GLM keeps what is good about linear regression, a **linear score** $\eta=\beta_0+\beta_1x_1+\cdots$, and adds two things: the **right distribution** for the outcome and a **link function** that bends the score into the allowed range ($\sigma(\eta)$ for probabilities, $e^\eta$ for rates). Each coefficient then has a clean meaning: *one more unit of $x_1$ multiplies the odds (or the rate) by $e^{\beta_1}$*.

## 4. Formal definitions

A GLM has (i) $y_i$ independent with a density in the **exponential family** with mean $\mu_i$ and variance $\phi\,V(\mu_i)$, (ii) a linear predictor $\eta_i=x_i^\top\beta$ (plus an offset), (iii) a link $g(\mu_i)=\eta_i$.

| Family | $V(\mu)$ | Canonical link $g$ | Inverse link $\mu$ | Use |
|---|---|---|---|---|
| Gaussian | 1 | identity | $\eta$ | continuous outcome |
| **Bernoulli / binomial** | $\mu(1-\mu)$ | **logit** $\ln\frac\mu{1-\mu}$ | $\frac1{1+e^{-\eta}}$ | yes/no outcome |
| **Poisson** | $\mu$ | **log** | $e^\eta$ | counts, rates |
| **Gamma** | $\mu^2$ | inverse (log is common) | $e^\eta$ | positive skewed amounts |
| Negative binomial | $\mu+\alpha\mu^2$ | log | $e^\eta$ | overdispersed counts |

**Score and information:** $U(\beta)=X^\top W\,\frac{y-\mu}{d\mu/d\eta}$, $\ \mathcal I(\beta)=X^\top WX/\phi$ with weights $W=\dfrac{(d\mu/d\eta)^2}{V(\mu)}$. **Deviance:** $D=2[\ell(\text{saturated})-\ell(\hat\beta)]$. **Offset:** for exposure $e_i$, $\ln\mu_i=\ln e_i+x_i^\top\beta$.

## 5. Derivations

### 5.1 IRLS is Newton's method
For the canonical link the score is $X^\top(y-\mu)$ and the Hessian is $-X^\top WX$ with $W=\mathrm{diag}\,V(\mu_i)$. Newton's step is $\beta_{\text{new}}=\beta+(X^\top WX)^{-1}X^\top(y-\mu)=(X^\top WX)^{-1}X^\top Wz$ with the **working response** $z=\eta+\frac{y-\mu}{d\mu/d\eta}$. **In plain words.** Each iteration solves a weighted least-squares problem on a corrected version of the data; the weights say how much each observation can be trusted at the current fit, and the loop stops when the coefficients stop moving (usually 4 to 8 rounds).

### 5.2 Logistic regression and odds ratios
With $p=\sigma(\eta)$, $\ \ln\frac p{1-p}=\beta_0+\beta_1x_1+\cdots$ is linear in $x$. Raising $x_1$ by one unit adds $\beta_1$ to the log-odds, i.e. multiplies the **odds** by $e^{\beta_1}$. **In plain words.** A coefficient of 0.7 means the odds are about doubled ($e^{0.7}=2.01$); it does not mean the probability doubles.

### 5.3 Poisson regression and rate ratios
With $\mu_i=e_i\exp(x_i^\top\beta)$: $\ \exp(\beta_j)$ is the **rate ratio** for one more unit of $x_j$. The score equations $\sum x_{ij}(y_i-\mu_i)=0$ say that observed and fitted totals match within every group defined by the covariates (including the intercept: $\sum y_i=\sum\mu_i$). **In plain words.** The model reproduces the total number of claims exactly, and the exposure offset converts "claims" into "claims per policy-year".

### 5.4 Standard errors
$\widehat{\mathrm{Var}}(\hat\beta)=\phi\,(X^\top WX)^{-1}$ at the fit, with $\phi=1$ for Bernoulli and Poisson and $\hat\phi=\frac1{n-p}\sum\frac{(y_i-\mu_i)^2}{V(\mu_i)}$ (Pearson) for the Gamma. A 95 % Wald interval is $\hat\beta\pm1.96\,\mathrm{SE}$, exponentiated for ratios. **In plain words.** Information is high where the weights are large: logistic regression learns most from observations whose probability is near 0.5 and little from near-certain ones.

### 5.5 Overdispersion
If the true variance is $\mu(1+\alpha\mu)$ but a Poisson model assumes $\mu$, the coefficient estimates remain consistent but the standard errors are too small by about $\sqrt{\hat\phi}$, with $\hat\phi=\chi^2_{\text{Pearson}}/(n-p)>1$. The negative binomial model adds the parameter $\alpha$ (a Gamma-mixed Poisson, 01_04/02_04). **In plain words.** If counts are more scattered than the Poisson allows (unobserved risk differences), the model is overconfident; widen the intervals or model the extra scatter.

## 6. Properties and relationships
- **Concave likelihood** for canonical links: IRLS converges to the unique MLE (unless the data are perfectly separated, when the logistic coefficients diverge).
- **Linear regression is the Gaussian-identity GLM**; the deviance is the residual sum of squares.
- **Link choice:** logit (odds ratios), probit (latent normal threshold), complementary log-log (asymmetric, time-to-event grouped data); they agree near $p=0.5$ and differ in the tails.
- **Rare events:** for small probabilities the odds ratio approximates the risk ratio (06_08).
- **Regularisation** adds a prior (09_12); **mixed models** add random effects for grouped data.
- **Link to ML and AI:** logistic regression is a one-layer neural network with sigmoid output, softmax regression its multi-class version, Poisson heads count outputs, and the GLM view explains loss and activation pairs (identity-MSE, sigmoid-BCE, softmax-CE, exp-Poisson).

## 7. A GLM fitter from scratch

One IRLS routine for all families and links, returning coefficients, standard errors, deviance and log-likelihood.

In [ ]:
from scipy.special import expit, gammaln
from scipy.optimize import minimize
LINKS = {"logit": (lambda m: np.log(m / (1 - m)), expit, lambda e: expit(e) * (1 - expit(e))),
         "probit": (stats.norm.ppf, stats.norm.cdf, stats.norm.pdf),
         "cloglog": (lambda m: np.log(-np.log(1 - m)), lambda e: 1 - np.exp(-np.exp(e)), lambda e: np.exp(e) * np.exp(-np.exp(e))),
         "log": (np.log, np.exp, np.exp), "identity": (lambda m: m, lambda e: e, lambda e: np.ones_like(e))}
FAM = {"binomial": lambda m: m * (1 - m), "poisson": lambda m: m, "gamma": lambda m: m ** 2, "gaussian": lambda m: np.ones_like(m)}
def glm(X, y, fam, link, offset=None, iters=50, tol=1e-10):
    n, p = X.shape; off = np.zeros(n) if offset is None else offset; g, ginv, dmu = LINKS[link]; V = FAM[fam]
    mu = (y + 0.5) / 2 if fam == "binomial" else np.maximum(y, 0.1) if fam in ("poisson", "gamma") else y.copy(); eta = g(mu); b = np.zeros(p)
    for it in range(iters):
        d = dmu(eta); z = eta - off + (y - mu) / d; w = d ** 2 / V(mu); A = X.T @ (X * w[:, None]); b_new = np.linalg.solve(A, X.T @ (w * z)); eta = X @ b_new + off; mu = ginv(eta)
        if np.max(np.abs(b_new - b)) < tol: b = b_new; break
        b = b_new
    d = dmu(eta); w = d ** 2 / V(mu); A = X.T @ (X * w[:, None]); phi = 1.0 if fam in ("binomial", "poisson") else np.sum((y - mu) ** 2 / V(mu)) / (n - p)
    if fam == "binomial": ll = np.sum(y * np.log(mu) + (1 - y) * np.log(1 - mu)); dev = -2 * ll
    elif fam == "poisson": ll = np.sum(stats.poisson.logpmf(y, mu)); dev = 2 * np.sum(np.where(y > 0, y * np.log(np.where(y > 0, y, 1) / mu), 0) - (y - mu))
    elif fam == "gamma": k = 1 / phi; ll = np.sum(stats.gamma.logpdf(y, k, scale=mu / k)); dev = 2 * np.sum(-np.log(y / mu) + (y - mu) / mu)
    else: ll = np.sum(stats.norm.logpdf(y, mu, np.sqrt(phi))); dev = np.sum((y - mu) ** 2)
    return dict(beta=b, se=np.sqrt(np.diag(np.linalg.inv(A)) * phi), mu=mu, eta=eta, dev=dev, ll=ll, phi=phi, iters=it + 1, cov=np.linalg.inv(A) * phi, p=p, n=n)
g = np.random.default_rng(0); x = g.normal(size=300); yb = (g.random(300) < expit(-0.5 + 1.2 * x)).astype(float); Xb = np.column_stack([np.ones(300), x]); fit = glm(Xb, yb, "binomial", "logit")
print("logistic fit:", fit["beta"].round(4), "SE", fit["se"].round(4), "iterations", fit["iters"]); print("score at the solution (should be ~0):", np.abs(Xb.T @ (yb - fit["mu"])).max())
ols = np.linalg.lstsq(Xb, g.normal(size=300), rcond=None)[0]; yg = 2 + 3 * x + g.normal(size=300); fg = glm(Xb, yg, "gaussian", "identity"); print("Gaussian GLM equals least squares:", fg["beta"].round(6), np.linalg.lstsq(Xb, yg, rcond=None)[0].round(6))
assert np.abs(Xb.T @ (yb - fit["mu"])).max() < 1e-6 and np.allclose(fg["beta"], np.linalg.lstsq(Xb, yg, rcond=None)[0], atol=1e-8)

## 8. 2D visuals

### 8.1 Link functions
The inverse links map the whole real line of scores into the allowed range: logit, probit and complementary log-log into $(0,1)$ (they agree near the middle and differ in the tails), and the exponential into $(0,\infty)$.

In [ ]:
e = np.linspace(-5, 5, 500); fig, axes = plt.subplots(1, 3, figsize=(17, 4.6)); axes[0].plot(e, expit(e), color=MAIN, label="logit"); axes[0].plot(e, stats.norm.cdf(e * 0.5882 * 1.0), color=HIGHLIGHT, label="probit (scaled)"); axes[0].plot(e, 1 - np.exp(-np.exp(e)), color=SIM, label="cloglog"); axes[0].legend(); axes[0].set_title("Links into (0, 1)"); axes[0].set_xlabel("linear predictor eta")
axes[1].plot(e, np.exp(e), color=MAIN); axes[1].set_ylim(0, 20); axes[1].set_title("Log link: mean = exp(eta) > 0"); axes[1].set_xlabel("eta")
pp = np.linspace(0.01, 0.99, 400); axes[2].plot(pp, np.log(pp / (1 - pp)), color=MAIN, label="logit"); axes[2].plot(pp, stats.norm.ppf(pp), color=HIGHLIGHT, label="probit"); axes[2].plot(pp, np.log(-np.log(1 - pp)), color=SIM, label="cloglog"); axes[2].set_ylim(-5, 5); axes[2].legend(); axes[2].set_title("Link functions g(p)"); axes[2].set_xlabel("probability p")
plt.show(); print("logit(0.5), probit(0.5), cloglog(0.5):", 0.0, 0.0, round(np.log(-np.log(0.5)), 4)); assert abs(expit(0.7) / (1 - expit(0.7)) - np.exp(0.7)) < 1e-12

### 8.2 Logistic regression: data, fit and the log-odds line
The fitted probability curve (left) and the same fit on the log-odds scale (right), where it is a straight line. Points are jittered.

In [ ]:
g = np.random.default_rng(1); n = 400; xs = g.normal(size=n); y_ = (g.random(n) < expit(-0.5 + 1.2 * xs)).astype(float); X_ = np.column_stack([np.ones(n), xs]); f_ = glm(X_, y_, "binomial", "logit"); xg = np.linspace(-3, 3, 200)
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8)); axes[0].scatter(xs, y_ + 0.04 * g.normal(size=n), s=8, color=NEUTRAL, alpha=0.6); axes[0].plot(xg, expit(f_["beta"][0] + f_["beta"][1] * xg), color=HIGHLIGHT, label="fitted P(y=1|x)"); axes[0].plot(xg, expit(-0.5 + 1.2 * xg), color="black", linestyle=":", label="truth"); axes[0].legend(); axes[0].set_xlabel("x"); axes[0].set_title("Probability scale")
bins_ = np.linspace(-2.5, 2.5, 11); mid = (bins_[:-1] + bins_[1:]) / 2; rate = np.array([y_[(xs >= a) & (xs < b)].mean() for a, b in zip(bins_[:-1], bins_[1:])]); ok = (rate > 0) & (rate < 1); axes[1].plot(mid[ok], np.log(rate[ok] / (1 - rate[ok])), "o", color=NEUTRAL, label="empirical log-odds by bin"); axes[1].plot(xg, f_["beta"][0] + f_["beta"][1] * xg, color=HIGHLIGHT, label="fitted line"); axes[1].legend(); axes[1].set_xlabel("x"); axes[1].set_title("Log-odds scale: a straight line")
plt.show(); print("fit", f_["beta"].round(3), "SE", f_["se"].round(3), "| odds ratio per unit x:", np.exp(f_["beta"][1]).round(3), "CI", np.exp(f_["beta"][1] + np.array([-1.96, 1.96]) * f_["se"][1]).round(3)); assert abs(f_["beta"][1] - 1.2) < 3 * f_["se"][1] and abs(f_["beta"][0] + 0.5) < 3 * f_["se"][0]

### 8.3 IRLS converges in a handful of iterations
The coefficient path of IRLS for the same logistic model, started from zero, and the log-likelihood increasing to its maximum.

In [ ]:
b = np.zeros(2); path = [b.copy()]; lls = []
for it in range(8):
    eta = X_ @ b; mu = expit(eta); w = mu * (1 - mu); z = eta + (y_ - mu) / w; b = np.linalg.solve(X_.T @ (X_ * w[:, None]), X_.T @ (w * z)); path.append(b.copy()); lls.append(np.sum(y_ * np.log(expit(X_ @ b)) + (1 - y_) * np.log(1 - expit(X_ @ b))))
path = np.array(path); fig, axes = plt.subplots(1, 2, figsize=(13, 4.4)); axes[0].plot(path[:, 0], label="intercept", color=MAIN, marker="o"); axes[0].plot(path[:, 1], label="slope", color=HIGHLIGHT, marker="o"); axes[0].legend(); axes[0].set_xlabel("IRLS iteration"); axes[0].set_title("Coefficients"); axes[1].plot(range(1, 9), lls, "o-", color=SIM); axes[1].set_xlabel("iteration"); axes[1].set_title("Log-likelihood")
plt.show(); print("differences between successive iterates:", np.abs(np.diff(path, axis=0)).max(axis=1).round(8)); assert np.abs(path[-1] - f_["beta"]).max() < 1e-8 and all(np.diff(lls) >= -1e-9)

## 9. 3D visuals

### 9.1 Static: the logistic log-likelihood over $(\beta_0,\beta_1)$
A concave dome with a single maximum at the MLE: this is why IRLS (Newton) converges so quickly.

In [ ]:
B0, B1 = np.meshgrid(np.linspace(-1.3, 0.4, 80), np.linspace(0.6, 2.0, 80)); LL = np.zeros_like(B0)
for i in range(80):
    for j in range(80): m_ = expit(B0[i, j] + B1[i, j] * xs); LL[i, j] = np.sum(y_ * np.log(m_) + (1 - y_) * np.log(1 - m_))
fig = plt.figure(figsize=(8.8, 6.2)); ax = fig.add_subplot(projection="3d"); ax.plot_surface(B0, B1, LL, cmap="viridis", edgecolor="none"); ax.scatter([f_["beta"][0]], [f_["beta"][1]], [f_["ll"]], color="red", s=50); ax.set_xlabel("intercept"); ax.set_ylabel("slope"); ax.set_zlabel("log-likelihood"); ax.set_title("Concave: one maximum at the MLE"); ax.view_init(elev=28, azim=-55)
plt.show(); i_ = np.unravel_index(LL.argmax(), LL.shape); print("grid maximum at", B0[i_].round(3), B1[i_].round(3), "| IRLS:", f_["beta"].round(3)); assert abs(B0[i_] - f_["beta"][0]) < 0.03 and abs(B1[i_] - f_["beta"][1]) < 0.03

### 9.2 Interactive Plotly version of 9.1

In [ ]:
fig = go.Figure(go.Surface(x=B0[0], y=B1[:, 0], z=LL, colorscale="Viridis")); fig.update_layout(title="Logistic log-likelihood", width=760, height=560, scene=dict(xaxis_title="intercept", yaxis_title="slope", zaxis_title="log-lik")); fig.show()

### 9.3 Static: a two-feature logistic regression surface
Probability of conversion as a function of two features (here: time on site and number of previous visits). The log-odds plane becomes an S-shaped sheet, steepest where $p=0.5$.

In [ ]:
A, Bm = np.meshgrid(np.linspace(0, 10, 80), np.linspace(0, 8, 80)); Pz = expit(-3.0 + 0.35 * A + 0.5 * Bm)
fig = plt.figure(figsize=(8.8, 6.2)); ax = fig.add_subplot(projection="3d"); ax.plot_surface(A, Bm, Pz, cmap="magma", edgecolor="none"); ax.set_xlabel("minutes on site"); ax.set_ylabel("previous visits"); ax.set_zlabel("P(convert)"); ax.set_title("A logistic regression is an S-shaped sheet"); ax.view_init(elev=26, azim=-130)
plt.show(); print("P at (0, 0), (5, 3), (10, 8):", Pz[0, 0].round(3), expit(-3 + 0.35 * 5 + 0.5 * 3).round(3), Pz[-1, -1].round(3)); assert Pz.min() > 0 and Pz.max() < 1

## 10. Simulation vs theory

### 10.1 Wald standard errors and coverage
Simulate 3,000 logistic datasets of $n=300$ and compare the standard deviation of $\hat\beta_1$ with the average Wald SE, and the coverage of the 95 % interval.

In [ ]:
g = np.random.default_rng(5); bh = []; se = []; cov = []
for _ in range(3000):
    xx = g.normal(size=300); yy = (g.random(300) < expit(-0.5 + 1.2 * xx)).astype(float); ff = glm(np.column_stack([np.ones(300), xx]), yy, "binomial", "logit"); bh.append(ff["beta"][1]); se.append(ff["se"][1]); cov.append(abs(ff["beta"][1] - 1.2) < 1.96 * ff["se"][1])
print(f"mean estimate {np.mean(bh):.4f} (truth 1.2) | SD of the estimates {np.std(bh):.4f} | mean Wald SE {np.mean(se):.4f} | coverage {np.mean(cov):.4f}"); assert abs(np.std(bh) / np.mean(se) - 1) < 0.06 and abs(np.mean(cov) - 0.95) < 0.015 and abs(np.mean(bh) - 1.2) < 0.03

### 10.2 Score equations and the Poisson total
For a Poisson regression with an intercept, the fitted counts add up to the observed counts exactly, and the deviance of a correct model is close to its degrees of freedom.

In [ ]:
g = np.random.default_rng(6); n2 = 2000; x2 = g.normal(size=n2); y2 = g.poisson(np.exp(0.5 + 0.4 * x2)); f2 = glm(np.column_stack([np.ones(n2), x2]), y2.astype(float), "poisson", "log")
print(f"sum of y = {y2.sum()}, sum of fitted = {f2['mu'].sum():.6f} | deviance {f2['dev']:.1f} on {n2 - 2} degrees of freedom | Pearson dispersion {np.sum((y2 - f2['mu']) ** 2 / f2['mu']) / (n2 - 2):.4f}")
assert abs(y2.sum() - f2["mu"].sum()) < 1e-6 and abs(np.sum((y2 - f2["mu"]) ** 2 / f2["mu"]) / (n2 - 2) - 1) < 0.08

## 11. `scipy.stats` and NumPy cheat sheet for this lesson

In [ ]:
print("logistic: expit(eta) = 1 / (1 + exp(-eta)) | odds ratio: np.exp(beta) | Poisson rate ratio: np.exp(beta) | offset: eta = X @ beta + np.log(exposure)")
print("deviance (Poisson): 2 * sum(y * log(y / mu) - (y - mu)) | Pearson dispersion: sum((y - mu) ** 2 / V(mu)) / (n - p) | Wald: beta +/- 1.96 * se")
print("likelihood-ratio test of nested models: stats.chi2.sf(2 * (ll_full - ll_reduced), df) | AIC = 2 p - 2 loglik")
print("library equivalent (statsmodels): sm.GLM(y, X, family=sm.families.Binomial()).fit() | sklearn LogisticRegression(penalty=None)")
print("check: 2 * (", round(f2["ll"], 2), "-", round(np.sum(stats.poisson.logpmf(y2, y2.mean())), 2), ") =", round(2 * (f2["ll"] - np.sum(stats.poisson.logpmf(y2, y2.mean()))), 2), "-> chi2(1) p =", stats.chi2.sf(2 * (f2["ll"] - np.sum(stats.poisson.logpmf(y2, y2.mean()))), 1))
assert f2["ll"] > np.sum(stats.poisson.logpmf(y2, y2.mean()))

## 12. Real-world worked examples

---
### Example 1 (main): predicting conversion with logistic regression

**Scenario.** An online shop records, for 5,000 visitors, the minutes on site ($x_1$, mean 5, sd 2.5, floored at 0) and whether they had visited before ($x_2\in\{0,1\}$, 40 % returning), and whether they bought. The true model is $\mathrm{logit}\,p=-3.4+0.35x_1+0.9x_2$.

**Questions**
1. Fit the logistic regression and report the coefficients, SEs and the odds ratios with 95 % intervals.
2. What is the predicted conversion probability for a new visitor with 8 minutes who has been before? For a first-time visitor with 2 minutes?
3. How many conversions does the model predict in total, and is the model calibrated by probability decile?
4. A linear probability model (least squares on 0/1) is fitted too: how many of its predictions fall outside $[0,1]$, and how far is it from the logistic probabilities?

### 12.1 By hand
Returning visitors have odds multiplied by $e^{0.9}=2.46$; each extra minute multiplies the odds by $e^{0.35}=1.42$. For 8 minutes and returning, $\eta=-3.4+0.35\cdot8+0.9=0.3$, so $p=\sigma(0.3)=0.574$. For a first-time visitor with 2 minutes, $\eta=-3.4+0.7=-2.7$, $p=0.063$.

In [ ]:
g = np.random.default_rng(21); n1 = 5000; t1 = np.maximum(g.normal(5, 2.5, n1), 0); ret = (g.random(n1) < 0.4).astype(float); pt = expit(-3.4 + 0.35 * t1 + 0.9 * ret); buy = (g.random(n1) < pt).astype(float)
X1 = np.column_stack([np.ones(n1), t1, ret]); f1 = glm(X1, buy, "binomial", "logit"); names = ["intercept", "minutes", "returning"]
tab = pd.DataFrame({"coef": f1["beta"], "SE": f1["se"], "odds ratio": np.exp(f1["beta"]), "OR low": np.exp(f1["beta"] - 1.96 * f1["se"]), "OR high": np.exp(f1["beta"] + 1.96 * f1["se"]), "true": [-3.4, 0.35, 0.9]}, index=names); print(f"conversion rate {buy.mean():.4f}, iterations {f1['iters']}"); print(tab.round(4).to_string())
p_a = expit(np.array([1, 8, 1]) @ f1["beta"]); p_b = expit(np.array([1, 2, 0]) @ f1["beta"]); print(f"Q2: P(8 min, returning) = {p_a:.4f} (truth {expit(0.3):.4f}); P(2 min, first-time) = {p_b:.4f} (truth {expit(-2.7):.4f})")
dec = np.digitize(f1["mu"], np.quantile(f1["mu"], np.linspace(0, 1, 11)[1:-1])); cal = pd.DataFrame({"predicted": [f1["mu"][dec == k].mean() for k in range(10)], "observed": [buy[dec == k].mean() for k in range(10)]}); print(f"Q3: predicted conversions {f1['mu'].sum():.1f} vs observed {buy.sum():.0f}"); print(cal.round(3).T.to_string())
lpm = X1 @ np.linalg.lstsq(X1, buy, rcond=None)[0]; print(f"Q4: linear probability model: {np.mean((lpm < 0) | (lpm > 1)):.4f} of predictions outside [0, 1] ({np.sum(lpm < 0)} below 0), largest gap to the logistic probability {np.abs(lpm - f1['mu']).max():.3f}, mean absolute gap {np.abs(lpm - f1['mu']).mean():.4f}")

### 12.2 Asserts

In [ ]:
assert all(abs(tab["coef"] - tab["true"]) < 3 * tab["SE"]) and abs(p_a - expit(0.3)) < 0.04 and abs(p_b - expit(-2.7)) < 0.02
assert abs(f1["mu"].sum() - buy.sum()) < 1e-6 and np.abs(cal["predicted"] - cal["observed"]).max() < 0.06 and np.mean(lpm < 0) > 0.01 and abs(np.exp(0.35) - 1.4191) < 1e-4 and abs(expit(0.3) - 0.5744) < 1e-4
print("all checks passed")

### 12.3 Verify by simulation
The Wald interval for the returning-visitor odds ratio should cover the true value $e^{0.9}$ in 95 % of repeated samples; the likelihood-ratio test of the null $\beta_{\text{returning}}=0$ rejects almost always at this sample size.

In [ ]:
g = np.random.default_rng(22); hit = 0; R = 400; lrs = []
for _ in range(R):
    t_ = np.maximum(g.normal(5, 2.5, 1500), 0); r_ = (g.random(1500) < 0.4).astype(float); y_s = (g.random(1500) < expit(-3.4 + 0.35 * t_ + 0.9 * r_)).astype(float); X_s = np.column_stack([np.ones(1500), t_, r_]); ff = glm(X_s, y_s, "binomial", "logit"); hit += abs(ff["beta"][2] - 0.9) < 1.96 * ff["se"][2]
    f0 = glm(X_s[:, :2], y_s, "binomial", "logit"); lrs.append(2 * (ff["ll"] - f0["ll"]))
print(f"coverage of the 95% Wald interval for the returning coefficient: {hit / R:.3f} | likelihood-ratio statistics: mean {np.mean(lrs):.1f}, share above the chi2(1) 95% point {np.mean(np.array(lrs) > 3.84):.3f}"); assert abs(hit / R - 0.95) < 0.04 and np.mean(np.array(lrs) > 3.84) > 0.95

### 12.4 Plots

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.8)); tg = np.linspace(0, 12, 100)
for r_, colr in [(0, MAIN), (1, HIGHLIGHT)]: axes[0].plot(tg, expit(f1["beta"][0] + f1["beta"][1] * tg + f1["beta"][2] * r_), color=colr, label="returning" if r_ else "first-time")
axes[0].scatter([8, 2], [p_a, p_b], color="black", zorder=3); axes[0].legend(); axes[0].set_xlabel("minutes on site"); axes[0].set_ylabel("P(buy)"); axes[0].set_title("Q2: fitted curves")
axes[1].plot([0, 0.5], [0, 0.5], color=NEUTRAL, linestyle=":"); axes[1].plot(cal["predicted"], cal["observed"], "o-", color=HIGHLIGHT); axes[1].set_xlabel("predicted (decile mean)"); axes[1].set_ylabel("observed rate"); axes[1].set_title("Q3: calibration")
axes[2].scatter(f1["mu"], lpm, s=3, color=NEUTRAL, alpha=0.5); axes[2].plot([0, 1], [0, 1], color=HIGHLIGHT); axes[2].axhline(0, color="black", linewidth=0.8); axes[2].set_xlim(0, 0.8); axes[2].set_xlabel("logistic probability"); axes[2].set_ylabel("linear probability model"); axes[2].set_title("Q4: the linear model goes below zero")
plt.show()

TBD

---
### Example 2: insurance claim rates with exposure (Poisson against negative binomial)

**Scenario.** 4,000 policies with exposure $e$ (policy-years, uniform 0.2 to 1), young driver (35 %) and urban (50 %). The claim rate per policy-year is $\exp(-2.2+0.6\,\text{young}+0.3\,\text{urban})$, but each policy has an unobserved risk multiplier $u\sim\mathrm{Gamma}(2,\text{scale }0.5)$ (mean 1), so the counts are overdispersed.

**Questions.** (1) Fit Poisson regression with the log-exposure offset: rate ratios and CIs. (2) Is there overdispersion (Pearson dispersion)? (3) Fit a negative binomial: how do the SEs change? (4) Over 300 repeated datasets, what is the coverage of the 95 % interval for the young coefficient under each model?

In [ ]:
def sim_ins(g, n=4000):
    e = g.uniform(0.2, 1, n); young = (g.random(n) < 0.35).astype(float); urban = (g.random(n) < 0.5).astype(float); u = g.gamma(2, 0.5, n); lam = e * u * np.exp(-2.2 + 0.6 * young + 0.3 * urban); return e, young, urban, g.poisson(lam)
def nb_fit(X, y, off):
    def nll(th):
        b, la = th[:-1], th[-1]; mu = np.exp(X @ b + off); k = 1 / np.exp(la); return -np.sum(stats.nbinom.logpmf(y, k, k / (k + mu)))
    r = minimize(nll, np.r_[np.zeros(X.shape[1]), 0.0], method="BFGS"); H = r.hess_inv; return r.x, np.sqrt(np.diag(H)), r.fun
g = np.random.default_rng(31); e2, yg_, ur, cl = sim_ins(g); X2 = np.column_stack([np.ones(len(e2)), yg_, ur]); fp = glm(X2, cl.astype(float), "poisson", "log", offset=np.log(e2)); pear = np.sum((cl - fp["mu"]) ** 2 / fp["mu"]) / (len(cl) - 3)
bnb, senb, nllnb = nb_fit(X2, cl, np.log(e2)); print(f"claims {cl.sum()}, policy-years {e2.sum():.0f}, overall rate {cl.sum() / e2.sum():.4f}")
print(pd.DataFrame({"Poisson coef": fp["beta"], "Poisson SE": fp["se"], "NB coef": bnb[:3], "NB SE": senb[:3], "true": [-2.2 + np.log(1), 0.6, 0.3]}, index=["intercept", "young", "urban"]).round(4).to_string())
print(f"Q1: rate ratio young {np.exp(fp['beta'][1]):.3f} (CI {np.exp(fp['beta'][1] - 1.96 * fp['se'][1]):.3f}, {np.exp(fp['beta'][1] + 1.96 * fp['se'][1]):.3f}), urban {np.exp(fp['beta'][2]):.3f} | Q2: Pearson dispersion {pear:.3f}; NB alpha = {np.exp(bnb[3]):.3f} (truth 0.5)")
print(f"Q3: SE ratio NB / Poisson: {(senb[:3] / fp['se']).round(3)} | log-likelihood Poisson {fp['ll']:.1f}, NB {-nllnb:.1f}, AIC {2 * 3 - 2 * fp['ll']:.1f} vs {2 * 4 + 2 * nllnb:.1f}")

### 12.6 Coverage over repeated datasets

In [ ]:
cov_p = []; cov_n = []; est = []
for _ in range(300):
    ee, yy_, uu, cc = sim_ins(g, 2000); XX = np.column_stack([np.ones(2000), yy_, uu]); a = glm(XX, cc.astype(float), "poisson", "log", offset=np.log(ee)); cov_p.append(abs(a["beta"][1] - 0.6) < 1.96 * a["se"][1]); est.append(a["beta"][1])
    bb, ss, _ = nb_fit(XX, cc, np.log(ee)); cov_n.append(abs(bb[1] - 0.6) < 1.96 * ss[1])
print(f"Q4: coverage of the 95% interval for the young coefficient: Poisson {np.mean(cov_p):.3f}, negative binomial {np.mean(cov_n):.3f} | mean estimate {np.mean(est):.3f}, SD of the estimates {np.std(est):.3f}")
assert pear > 1.2 and np.all(senb[:3] > fp["se"]) and 0.4 < np.exp(bnb[3]) < 0.65 and np.mean(cov_p) < 0.92 and abs(np.mean(cov_n) - 0.95) < 0.04 and abs(np.mean(est) - 0.6) < 0.05
fig, axes = plt.subplots(1, 3, figsize=(17, 4.6)); lam = fp["mu"]; axes[0].hist(cl, bins=np.arange(-0.5, 8.5, 1), density=True, color=NEUTRAL, alpha=0.8, label="observed claims"); axes[0].plot(np.arange(8), [np.mean(stats.poisson.pmf(k, lam)) for k in range(8)], "o-", color=MAIN, label="Poisson fit"); kk = 1 / np.exp(bnb[3]); mu_nb = np.exp(X2 @ bnb[:3] + np.log(e2)); axes[0].plot(np.arange(8), [np.mean(stats.nbinom.pmf(k, kk, kk / (kk + mu_nb))) for k in range(8)], "o-", color=HIGHLIGHT, label="negative binomial"); axes[0].legend(); axes[0].set_xlabel("claims"); axes[0].set_title("Q2: the Poisson misses the tail")
axes[1].bar(["Poisson", "neg. binomial"], [np.mean(cov_p), np.mean(cov_n)], color=[MAIN, HIGHLIGHT]); axes[1].axhline(0.95, color="black", linestyle=":"); axes[1].set_ylim(0.6, 1); axes[1].set_ylabel("coverage of the 95% interval"); axes[1].set_title("Q4: coverage over 300 datasets")
axes[2].errorbar([0, 1], [fp["beta"][1], bnb[1]], yerr=[1.96 * fp["se"][1], 1.96 * senb[1]], fmt="o", color=HIGHLIGHT, capsize=5); axes[2].axhline(0.6, color="black", linestyle=":"); axes[2].set_xticks([0, 1]); axes[2].set_xticklabels(["Poisson", "neg. binomial"]); axes[2].set_title("Q3: same estimate, wider honest interval")
plt.show()

TBD

---
### Example 3: claim severity with a Gamma GLM (and why not log-transform)

**Scenario.** Claim amounts for 3,000 claims follow a Gamma distribution with shape 2 and mean $\exp(6+0.5\,\text{vehicle\_age\_group})$ (vehicle group 0, 1, 2; scale = mean/2). We want the **expected claim cost** per group.

**Questions.** (1) Fit a Gamma GLM with log link and report the multiplicative effect per group with its SE and the estimated dispersion $\hat\phi$ (truth $1/2$). (2) Compare with least squares on $\ln y$ followed by $\exp$: how biased is the predicted *mean* cost? (3) What does the Gamma GLM say about the coefficient of variation?

In [ ]:
g = np.random.default_rng(41); n3 = 3000; grp = g.integers(0, 3, n3).astype(float); mu3 = np.exp(6 + 0.5 * grp); y3 = g.gamma(2, mu3 / 2); X3 = np.column_stack([np.ones(n3), grp]); fg_ = glm(X3, y3, "gamma", "log")
ols_log = np.linalg.lstsq(X3, np.log(y3), rcond=None)[0]; resid_var = np.var(np.log(y3) - X3 @ ols_log, ddof=2)
rows = []
for k in range(3): rows.append({"group": k, "true mean": mu3[grp == k][0], "observed mean": y3[grp == k].mean(), "Gamma GLM": np.exp(fg_["beta"][0] + fg_["beta"][1] * k), "exp(OLS on log)": np.exp(ols_log[0] + ols_log[1] * k), "OLS with variance correction": np.exp(ols_log[0] + ols_log[1] * k + resid_var / 2)})
tb3 = pd.DataFrame(rows).set_index("group"); print(f"Q1: coefficients {fg_['beta'].round(4)} (truth 6, 0.5), SE {fg_['se'].round(4)}, dispersion {fg_['phi']:.4f} (truth 0.5), iterations {fg_['iters']}"); print(tb3.round(1).to_string())
print(f"Q2: bias of exp(OLS on log): {(tb3['exp(OLS on log)'] / tb3['true mean'] - 1).round(3).to_numpy()}, of the Gamma GLM: {(tb3['Gamma GLM'] / tb3['true mean'] - 1).round(3).to_numpy()} | Q3: coefficient of variation sqrt(phi) = {np.sqrt(fg_['phi']):.3f} (truth {np.sqrt(0.5):.3f})")

### 12.7 Asserts

In [ ]:
assert abs(fg_["beta"][0] - 6) < 3 * fg_["se"][0] and abs(fg_["beta"][1] - 0.5) < 3 * fg_["se"][1] and abs(fg_["phi"] - 0.5) < 0.04
assert (tb3["exp(OLS on log)"] / tb3["true mean"] - 1).abs().min() > 0.2 and (tb3["Gamma GLM"] / tb3["true mean"] - 1).abs().max() < 0.06 and (tb3["OLS with variance correction"] / tb3["true mean"] - 1).abs().max() < 0.15
print("all checks passed")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.6)); xg = np.linspace(0, 6000, 400); colr3 = [MAIN, SIM, HIGHLIGHT]
for k in range(3): axes[0].hist(y3[grp == k], bins=60, density=True, histtype="step", color=colr3[k], range=(0, 6000)); axes[0].plot(xg, stats.gamma.pdf(xg, 1 / fg_["phi"], scale=tb3.loc[k, "Gamma GLM"] * fg_["phi"]), color=colr3[k], linestyle="--")
axes[0].set_xlabel("claim amount"); axes[0].set_title("Q1: Gamma fits by group (dashed)"); w = 0.25; axes[1].bar(np.arange(3) - w, tb3["true mean"], w, color=NEUTRAL, label="true"); axes[1].bar(np.arange(3), tb3["Gamma GLM"], w, color=HIGHLIGHT, label="Gamma GLM"); axes[1].bar(np.arange(3) + w, tb3["exp(OLS on log)"], w, color=MAIN, label="exp(OLS on log y)"); axes[1].legend(fontsize=8); axes[1].set_xlabel("vehicle group"); axes[1].set_title("Q2: expected claim cost")
axes[2].scatter(fg_["mu"], (y3 - fg_["mu"]) / fg_["mu"], s=3, color=NEUTRAL, alpha=0.5); axes[2].axhline(0, color="black"); axes[2].set_xlabel("fitted mean"); axes[2].set_ylabel("relative residual (y - mu) / mu"); axes[2].set_title("Spread proportional to the mean (Gamma)")
plt.show()

TBD

---
### Example 4: which link? logit, probit or complementary log-log

**Scenario.** A pesticide trial exposes 12 batches of 200 insects to doses from 0.5 to 6 (log-scale). The true mortality curve is *asymmetric*: $P(\text{death})=1-\exp(-\exp(-4+1.8\ln(\text{dose}\cdot2)))$ (a complementary log-log curve). We fit the three links and compare.

**Questions.** (1) Which link has the lowest deviance and AIC? (2) How different are the predicted mortalities at low (dose 1) and high (dose 5) doses? (3) Estimate the dose that kills 50 % (LD50) from each model and compare with the truth.

In [ ]:
doses = np.linspace(0.5, 6, 12); ld = np.log(doses); m_true = lambda d: 1 - np.exp(-np.exp(-4 + 1.8 * np.log(d * 2))); g = np.random.default_rng(51); N = 200; k_ = g.binomial(N, m_true(doses)); rows = []
Xd = np.column_stack([np.ones(12), ld]); fits = {}
def glm_binom_grouped(X, k, N, link):
    g_, ginv, dmu = LINKS[link]; b = np.zeros(X.shape[1]); p0 = (k + 0.5) / (N + 1); eta = g_(p0); mu = p0
    for _ in range(100):
        d = dmu(eta); z = eta + (k / N - mu) / d; w = N * d ** 2 / (mu * (1 - mu)); bn = np.linalg.solve(X.T @ (X * w[:, None]), X.T @ (w * z)); eta = X @ bn; mu = ginv(eta)
        if np.max(np.abs(bn - b)) < 1e-12: b = bn; break
        b = bn
    ll = np.sum(stats.binom.logpmf(k, N, mu)); sat = np.sum(stats.binom.logpmf(k, N, k / N)); return dict(beta=b, mu=mu, ll=ll, dev=2 * (sat - ll), aic=2 * X.shape[1] - 2 * ll)
for lk in ["logit", "probit", "cloglog"]: fits[lk] = glm_binom_grouped(Xd, k_, N, lk)
ld50 = {lk: np.exp((LINKS[lk][0](0.5) - f["beta"][0]) / f["beta"][1]) for lk, f in fits.items()}; true_ld50 = np.exp((np.log(-np.log(0.5)) + 4) / 1.8) / 2
print(pd.DataFrame({lk: {"deviance (10 df)": f["dev"], "AIC": f["aic"], "slope": f["beta"][1], "LD50": ld50[lk]} for lk, f in fits.items()}).T.round(3).to_string()); print(f"true LD50 = {true_ld50:.4f}")
for d_ in (1, 5):
    print(f"dose {d_}: true {m_true(d_):.4f}", {lk: round(float(LINKS[lk][1](f['beta'][0] + f['beta'][1] * np.log(d_))), 4) for lk, f in fits.items()})

### 12.8 Asserts

In [ ]:
assert fits["cloglog"]["aic"] < fits["logit"]["aic"] and fits["cloglog"]["aic"] < fits["probit"]["aic"] and abs(ld50["cloglog"] - true_ld50) < 0.08 * true_ld50
assert fits["cloglog"]["dev"] < stats.chi2.ppf(0.99, 10)
print("all checks passed")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8)); dg = np.linspace(0.4, 6.2, 200); axes[0].plot(doses, k_ / N, "o", color="black", label="observed"); axes[0].plot(dg, m_true(dg), color="gray", linestyle=":", label="truth")
for lk, colr in zip(fits, [MAIN, SIM, HIGHLIGHT]): axes[0].plot(dg, LINKS[lk][1](fits[lk]["beta"][0] + fits[lk]["beta"][1] * np.log(dg)), color=colr, label=lk)
axes[0].legend(); axes[0].set_xlabel("dose"); axes[0].set_ylabel("mortality"); axes[0].set_title("Q1-Q2: fits with three links")
for lk, colr in zip(fits, [MAIN, SIM, HIGHLIGHT]): axes[1].plot(doses, (k_ / N - fits[lk]["mu"]) / np.sqrt(fits[lk]["mu"] * (1 - fits[lk]["mu"]) / N), "o-", color=colr, label=lk)
axes[1].axhline(0, color="black"); axes[1].legend(); axes[1].set_xlabel("dose"); axes[1].set_ylabel("Pearson residual"); axes[1].set_title("The symmetric links leave a pattern")
plt.show()

TBD

## 13. Where it is used

### Statistics
- **Logistic regression** for binary outcomes in medicine and the social sciences (odds ratios), **Poisson and negative binomial regression** for incidence rates and epidemiology (offsets for person-time), **Gamma and Tweedie GLMs** in insurance pricing, **survival models** as Poisson GLMs, **mixed models** (GLMMs).

### Data science
- **Conversion, churn and default models** (logit), **demand and claim-count models** (Poisson, NB), **severity and cost models** (Gamma), interpretability through odds and rate ratios, feature effects for pricing and risk rate filings, strong baselines before gradient boosting.

### ML and AI
- **Logistic and softmax regression are single-layer neural networks;** the GLM pairing (activation, loss) = (identity, MSE), (sigmoid, BCE), (softmax, CE), (exp, Poisson) is exactly the choice of output layer, and the gradient $y-\mu$ is why they train well. GLMs are also the linear heads of many deep models and the base of **GAMs** and **calibration** methods.

## 14. Common pitfalls and misconceptions

- **Reading a logit coefficient as a change in probability:** it changes the log-odds; the effect on probability depends on where you start (steepest near 0.5).
- **Odds ratio $\neq$ risk ratio** when the outcome is common (conversion at 30 %); only for rare events do they match (06_08).
- **Forgetting the offset** in rate models: without $\ln(\text{exposure})$ a policy observed for 2 months is treated like one observed for a year.
- **Ignoring overdispersion:** Poisson standard errors are too small when the dispersion exceeds 1 (Example 2); check Pearson $\chi^2/\mathrm{df}$ and consider the negative binomial or quasi-Poisson.
- **Perfect separation:** if a feature separates the classes, logistic coefficients diverge; add regularisation (09_12).
- **Back-transforming a log model naively:** $\exp(E\ln y)$ estimates the geometric mean, not the mean; use a Gamma GLM or a variance correction (Example 3).
- **Linear probability model outside $[0,1]$:** fine as a quick summary near the middle, not for predictions (Example 1).
- **Using the wrong link by habit:** compare links by deviance and AIC when the response curve may be asymmetric (Example 4).

## 15. Exercises

1. A logistic model has $\mathrm{logit}\,p=-2+0.5x$. Compute $p$ for $x=0,4,8$ and the odds ratio and the change in probability for $x$ from 4 to 5.
2. A Poisson model has $\ln\lambda=-1+0.7x$ per policy-year. For $x=1$ and an exposure of 0.25 years compute the expected claims and the probability of no claim.
3. In Example 1, how many extra minutes on site give the same odds increase as being a returning visitor?
4. Compute the likelihood-ratio statistic for the returning-visitor coefficient in Example 1 and its p-value.
5. Show numerically that the Poisson GLM score equations hold: $\sum x_{ij}(y_i-\hat\mu_i)=0$ for all $j$ in Example 2.
6. (Harder) Fit a logistic regression with an interaction between minutes and returning (Example 1 data) and test whether the interaction improves the fit.

Solutions are at the end.

## 16. Summary table

| Model | Family / link | Coefficient means | Check |
|---|---|---|---|
| Linear | Gaussian / identity | change in mean | residuals |
| Logistic | Bernoulli / logit | $e^\beta$ = odds ratio | calibration |
| Poisson | Poisson / log | $e^\beta$ = rate ratio (offset = exposure) | dispersion |
| Negative binomial | NB / log | rate ratio, extra scatter $\alpha$ | dispersion |
| Gamma | Gamma / log | multiplicative effect on the mean | CV constant |
| IRLS | weighted least squares loop | Newton's method | $X^\top(y-\mu)=0$ |
| tools | `glm` (this notebook), statsmodels | | |

## Solutions

In [ ]:
print("1: p(0, 4, 8) =", expit(np.array([-2, 0, 2.0])).round(4), "| OR per unit", round(np.exp(0.5), 4), "| P(x = 5) - P(x = 4) =", round(expit(0.5) - expit(0), 4))
lam = 0.25 * np.exp(-1 + 0.7); print(f"2: expected claims {lam:.4f}, P(no claim) = {np.exp(-lam):.4f}")
print(f"3: {0.9 / 0.35:.2f} extra minutes (fitted: {f1['beta'][2] / f1['beta'][1]:.2f})")
f0_ = glm(X1[:, [0, 1]], buy, "binomial", "logit"); lr = 2 * (f1["ll"] - f0_["ll"]); print(f"4: LR statistic {lr:.1f}, p = {stats.chi2.sf(lr, 1):.3e}")
print("5: score residuals:", np.abs(X2.T @ (cl - fp["mu"])).max().round(6))
Xi = np.column_stack([X1, t1 * ret]); fi = glm(Xi, buy, "binomial", "logit"); lri = 2 * (fi["ll"] - f1["ll"]); print(f"6: interaction coefficient {fi['beta'][3]:.4f} (SE {fi['se'][3]:.4f}), LR = {lri:.2f}, p = {stats.chi2.sf(lri, 1):.3f} (the truth has no interaction)")
assert np.abs(X2.T @ (cl - fp["mu"])).max() < 1e-5

TBD